# LangChain Expression Language (LCEL)
### Compose chains with `|`, run them in parallel, and branch on conditions

---
**Topics Covered:**
- The `|` pipe operator — connecting runnables
- `RunnableParallel` — run independent chains at the same time
- `RunnableLambda` — wrap any Python function as a runnable
- `RunnablePassthrough` — forward inputs unchanged
- `RunnableBranch` — conditional routing
- Inspecting chain schemas with `.input_schema` and `.output_schema`


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough, RunnableBranch

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.5)

## 1. The Pipe Operator `|`

Every LangChain component that implements `Runnable` can be chained with `|`.  
Data flows left → right through each step.

In [ ]:
# A simple 3-step chain
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical summariser. Be concise."),
    ("human", "Summarise the following topic in exactly 3 bullet points:\n\n{topic}")
])

summarize_chain = summarize_prompt | llm | StrOutputParser()

output = summarize_chain.invoke({"topic": "Transformer architecture in deep learning"})
print(output)

## 2. `RunnableLambda` — Python Functions as Runnables

In [ ]:
# Post-processing step: count words in the output
word_counter = RunnableLambda(lambda text: {
    "summary": text,
    "word_count": len(text.split())
})

# Extend the chain with a custom Python step
extended_chain = summarize_chain | word_counter

result = extended_chain.invoke({"topic": "Gradient descent optimisation"})
print("Word count:", result["word_count"])
print("Summary:\n",  result["summary"])

## 3. `RunnablePassthrough` — Pass Input Alongside

Use `RunnablePassthrough.assign(key=chain)` to add new fields to a dict while keeping original fields intact.

In [ ]:
# Keep the original 'topic' AND add the LLM's summary
augment_chain = (
    RunnablePassthrough.assign(
        summary=summarize_chain
    )
)

out = augment_chain.invoke({"topic": "RLHF — reinforcement learning from human feedback"})
print("Original input:", out["topic"])
print("\nAdded summary:\n", out["summary"])

## 4. `RunnableParallel` — Fan-Out to Multiple Chains

Run independent chains concurrently and collect their outputs in a single dict.

In [ ]:
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List only PROS. Use 3 bullet points max."),
    ("human", "Topic: {topic}")
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List only CONS. Use 3 bullet points max."),
    ("human", "Topic: {topic}")
])

pros_chain = pros_prompt | llm | StrOutputParser()
cons_chain = cons_prompt | llm | StrOutputParser()

# Both chains receive the same input and run concurrently
pros_cons = RunnableParallel(pros=pros_chain, cons=cons_chain)

result = pros_cons.invoke({"topic": "Microservices architecture"})
print("PROS:\n", result["pros"])
print("\nCONS:\n", result["cons"])

## 5. Chaining Parallel Results into a Final Step

In [ ]:
combine_prompt = ChatPromptTemplate.from_messages([
    ("system", "You synthesise pros/cons into an actionable recommendation."),
    ("human", "PROS:\n{pros}\n\nCONS:\n{cons}\n\nGive a 2-sentence recommendation.")
])

full_pipeline = pros_cons | combine_prompt | llm | StrOutputParser()

recommendation = full_pipeline.invoke({"topic": "Microservices architecture"})
print(recommendation)

## 6. `RunnableBranch` — Conditional Routing

Route to different chains based on a condition evaluated on the input.

In [ ]:
# Tone-aware responder: formal vs casual based on 'tone' key
formal_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional assistant. Respond formally."),
    ("human", "{question}")
])

casual_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly chatbot. Respond casually with emojis."),
    ("human", "{question}")
])

formal_chain = formal_prompt | llm | StrOutputParser()
casual_chain = casual_prompt | llm | StrOutputParser()

tone_branch = RunnableBranch(
    (lambda x: x.get("tone") == "formal",  formal_chain),
    casual_chain    # default branch
)

print("--- Formal ---")
print(tone_branch.invoke({"question": "What is recursion?", "tone": "formal"}))

print("\n--- Casual ---")
print(tone_branch.invoke({"question": "What is recursion?", "tone": "casual"}))

## 7. Inspect Chain Schemas

In [ ]:
import json
print("Input schema:")
print(json.dumps(summarize_chain.input_schema.model_json_schema(), indent=2))

print("\nOutput schema:")
print(json.dumps(summarize_chain.output_schema.model_json_schema(), indent=2))

## 8. Chain Introspection with `.get_graph()`

In [ ]:
graph = full_pipeline.get_graph()
graph.print_ascii()

---
### Summary

| Runnable | What it does |
|----------|--------------|
| `A \| B \| C` | Sequential pipeline |
| `RunnableLambda(fn)` | Any Python function as a step |
| `RunnablePassthrough.assign(key=chain)` | Augment input dict with new field |
| `RunnableParallel(a=chain1, b=chain2)` | Run concurrently, merge outputs |
| `RunnableBranch((cond, chain), default)` | Conditional routing |

**Next → `05_streaming_and_batch.ipynb`**